# The Body of the Heart: Geometry Generation

Before we can simulate the physics of the heart, we must first define its shape. In the field of computational cardiac mechanics, the choice of domain—the digital body of our virtual patient is very imporant. We generally face a trade-off between anatomical realism and experimental control. Patient-specific geometries derived from medical imaging offer high fidelity but introduce significant complexity and variability that can obscure mechanical relationships. Conversely, idealized geometries derived from mathematical primitives provide a clean, controlled environment.

For this thesis, we have chosen to work with idealized bi-ventricular geometies. By using a mathematically defined shape, we gain precise parametric control over critical variables such as wall thickness, curvature, and cavity volume. This allows us to rigorously test our hypotheses regarding ventricular interaction without the confounding noise of individual anatomical idiosyncrasies.

## The Truncated Ellipsoid

We approximate the Left Ventricle (LV) and Right Ventricle (RV) using a truncated ellipsoid model. This shape is sophisticated enough to capture the essential topology of the heart—specifically the way the thin-walled RV wraps around the thick-walled LV—yet remains simple enough to be described by a manageable set of parameters.

The result is a clean, high-quality finite element mesh free from the imaging artifacts and jagged edges that often plague patient-specific models. This ensures that any stress concentrations we observe in our simulations are genuine physical phenomena, not numerical errors caused by a poor mesh.

In [ ]:
# --- Setup & Imports (Click to Expand) ---
# We use the `cardiac-geometries` library to handle the complex
# mesh generation logic. This library uses Gmsh under the hood to
# create high-quality tetrahedral meshes from our parameters.

from pathlib import Path
from mpi4py import MPI
import cardiac_geometries

# We will save our generated mesh here
geodir = Path("biv_geometry")
if not geodir.exists():
    geodir.mkdir(parents=True)

## Microstructure: The Fiber Architecture

Defining the gross anatomy is only the first step. The heart is not a homogeneous rubber balloon; it is a complex biological composite with a highly specific grain. The myocardium is **orthotropic**, which means its stiffness and contraction strength depend heavily on the direction of the underlying muscle fibers. Accurately capturing this microstructure is essential for a realistic simulation.

To model this, we assign a local fiber direction vector, $f_0$, to every point within our mesh. The orientation of these fibers is not constant. Instead, the fiber angle $\alpha$ (measured relative to the circumferential plane) rotates linearly as we move through the wall from the outside to the inside. This relationship is described by the following equation:

$$
\alpha(d) = \alpha_{endo} (1 - d) + \alpha_{epi} d
$$

Here, $d$ represents the normalized transmural distance, which is zero at the endocardium and one at the epicardium. In our model, we adopt standard physiological values where the fibers rotate from $+60^\circ$ at the inner wall to $-60^\circ$ at the outer wall. This transmural variation creates the famous **"double-helix" structure** of the heart muscle, a functional architecture that allows the heart to twist during contraction and wring out blood with remarkable efficiency.

In [ ]:
# Generate the BiV ellipsoid mesh with fibers
cardiac_geometries.mesh.biv_ellipsoid(
    outdir=geodir,
    create_fibers=True,          # Automatically compute fiber directions using LDRB
    fiber_space="Quadrature_6",  # High-order interpolation for smooth fiber fields
    comm=MPI.COMM_WORLD,         # Run in parallel
    char_length=0.5,             # Controls the fineness of the mesh (smaller = finer)
    fiber_angle_epi=-60,         # Fiber angle at the epicardium
    fiber_angle_endo=60,         # Fiber angle at the endocardium
)

## Anatomy of the Digital Heart

The generation process yields more than just a cloud of points and a fiber field. It also produces a set of semantically meaningful **tags**, or markers, that identify the distinct anatomical regions of our model. These tags are the bridge between our geometry and our physics solvers. They allow us to apply the correct boundary conditions to specific surfaces—such as applying blood pressure only to the inner walls or attaching the pericardial constraint only to the outer surface.

Specifically, our mesh contains four key surfaces. The **ENDO_LV** (ID 1) and **ENDO_RV** (ID 2) markers define the inner surfaces of the left and right ventricles, respectively, where the pressure loads $P_{LV}(t)$ and $P_{RV}(t)$ will be applied. The **EPI** (ID 3) marker identifies the epicardium, where we will enforce the pericardial spring constraint. Finally, the **BASE** (ID 4) marker defines the top cut plane, where we apply elastic supports to mimic the connective tissue that holds the heart in the chest.

This structured tagging system is essential for the flexibility of our computational pipeline. It allows us to write generic solver code that can operate on *any* heart geometry—whether it is our idealized ellipsoid or a complex patient-specific mesh—provided it contains these four standard surface definitions.

:::{seealso}
For a deeper exploration of the mathematics behind fiber assignment, specifically how the **Laplace-Dirichlet Rule-Based (LDRB)** algorithm maps these angles to arbitrary shapes, please refer to the **[Foundational Theory: Fibers](../05_theory/fibers.md)** section.
:::